# Lennard-Jones Fluid Coexistence via fvGEMC

Computes liquid–vapor coexistence densities for the truncated Lennard-Jones fluid with the
Fixed-Volume variant of Gibbs Ensemble Monte Carlo (fvGEMC): a set of short, fixed-volume
GEMC runs, with results fit to recover the coexistence densities, just like regular GEMC.

1. **Simulate** — run fvGEMC for a set of (initial density, temperature) pairs.
2. **Fit** — reshape results into one finite-volume family per temperature, then fit to recover the true coexistence densities.
3. **Compare** — plot against a direct, long-run reference GEMC simulation.

Setup, GEMC engine, and Fitting engine below are collapsed by default — they're reusable
library code, not required reading to follow the analysis.

## Setup

This notebook needs a Python 3 kernel to execute. Colab (and any well-behaved notebook host)
picks it up automatically; if yours doesn't, select a **Python 3** kernel manually.

Run this section once to install the required packages (`numpy`, `scipy`, `matplotlib`, and
`numba` — the last one JIT-compiles the GEMC engine's inner Monte Carlo loop, since a plain
Python loop over the ~5×10⁶ steps in a single simulation would otherwise be far too slow) and
prepare the kernel, then run the rest of the notebook normally.

Everything runs inside this notebook except package installation, which needs network access.

In [ ]:
import importlib.util, subprocess, sys

for pkg in ["numpy", "scipy", "matplotlib", "numba"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pkg], check=True)

import math, os, tempfile
from types import SimpleNamespace
import numpy as np
from scipy.optimize import curve_fit, minimize
import matplotlib.pyplot as plt
from numba import njit

In [ ]:
plt.switch_backend("Agg")   # headless plotting (no display server)
SCRATCHDIR = tempfile.mkdtemp()   # per-run input files for the GEMC engine's file-based interface;
                                   # nothing produced by a run needs to persist, so this is the only path we need

print("CPUs available:", os.cpu_count(), " (simulations run sequentially -- see the note in Section 1)")

## GEMC engine

A Gibbs Ensemble Monte Carlo engine for the truncated (`rc=3σ`) Lennard-Jones fluid: particle
translation, volume-exchange, and particle-swap moves between two boxes, with adaptively-tuned
step sizes (the hot inner loop is JIT-compiled with numba for speed). `main(infile, outfile,
posfile)` runs one simulation and returns the dilute and dense box densities (`rho1`, `rho2`),
averaged over the second half of the run (post-equilibration). The same engine produced the
reference GEMC densities below, in a separate, longer run; those results are precomputed and
hardcoded rather than generated live here.

In [ ]:
# Minimum-image convention: box coordinates are stored in [-0.5, 0.5); wrap()
# folds a coordinate difference back into that range so a distance never exceeds L/2.
@njit
def wrap(x):
    return x - round(x)

# A simple linear congruential generator, ported from the Julia notebook's custom
# per-thread LCG. The simulations here run sequentially (see Section 1), so a single
# state slot suffices; state is a length-1 int64 array so it can be mutated in place
# and threaded through numba-jitted functions by reference.
LCG_A = np.int64(1103515245)
LCG_C = np.int64(12345)
LCG_MASK = np.int64(0x7fffffff)
LCG_NORM = 2147483647.0

@njit
def lcg_uniform(state):
    state[0] = (state[0] * LCG_A + LCG_C) & LCG_MASK
    return state[0] / LCG_NORM

@njit
def lcg_bool(state):
    return lcg_uniform(state) < 0.5

@njit
def lcg_randrange(state, n):
    # Uniform integer in [0, n).
    return min(int(lcg_uniform(state) * n), n - 1)

# ── Box ──

class Box:
    """particle positions (pos: fractional box coordinates in [-0.5, 0.5); only rows
    [0:n) are live -- pos is preallocated to ntotal capacity, since swap moves can in
    principle move every particle into one box), n (particle count), L (box side
    length), rc2 (squared cutoff radius, real units, same scale as L, not fractional),
    erc (LJ energy at the cutoff, subtracted so the potential is continuous there), js
    (translation step size, real units, adaptively tuned by adjust_step_core), ntry
    (translation move attempts since the last adjust_step_core call), naccept
    (translation move acceptances since the last adjust_step_core call)."""
    __slots__ = ("pos", "n", "L", "rc2", "erc", "js", "ntry", "naccept")

    def __init__(self, pos, n, L, rc2, erc, js=0.005, ntry=0, naccept=0):
        self.pos, self.n, self.L = pos, n, L
        self.rc2, self.erc = rc2, erc
        self.js, self.ntry, self.naccept = js, ntry, naccept

    def V(self):
        return self.L ** 3

    def rho(self):
        return self.n / self.V()

# ── Energy ──

@njit
def lj(dx, dy, dz, L, rc2, erc):
    dx = wrap(dx); dy = wrap(dy); dz = wrap(dz)
    r2 = (dx * dx + dy * dy + dz * dz) * L * L
    if r2 > rc2:
        return 0.0
    r6 = 1.0 / (r2 * r2 * r2)
    return 4.0 * r6 * (r6 - 1.0) - erc

@njit
def particle_energy(pos, n, px, py, pz, skip, L, rc2, erc):
    e = 0.0
    for j in range(n):
        if j == skip:
            continue
        e += lj(px - pos[j, 0], py - pos[j, 1], pz - pos[j, 2], L, rc2, erc)
    return e

@njit
def total_energy_core(pos, n, L, rc2, erc):
    e = 0.0
    for i in range(n - 1):
        for j in range(i + 1, n):
            e += lj(pos[i, 0] - pos[j, 0], pos[i, 1] - pos[j, 1], pos[i, 2] - pos[j, 2], L, rc2, erc)
    return e

def total_energy(b):
    return total_energy_core(b.pos, b.n, b.L, b.rc2, b.erc)

# ── Lattice init ──

def make_box(N, L, rc, capacity):
    rc6 = (1.0 / rc) ** 6
    erc = 4 * rc6 * (rc6 - 1)
    pos = np.zeros((capacity, 3))
    n = math.ceil(N ** (1.0 / 3.0))
    idx = 0
    done = False
    for i in range(n):
        if done:
            break
        for j in range(n):
            if done:
                break
            for k in range(n):
                if idx == N:
                    done = True
                    break
                pos[idx, 0] = (i + 0.5) / n - 0.5
                pos[idx, 1] = (j + 0.5) / n - 0.5
                pos[idx, 2] = (k + 0.5) / n - 0.5
                idx += 1
    return Box(pos, idx, L, rc ** 2, erc, 0.005, 0, 0)

# ── Translation move ──

@njit
def adjust_step_core(js, ntry, naccept):
    ratio = naccept / ntry
    if ratio < 0.5:
        if js > 1e-6:
            js *= 0.99
    else:
        if js < 1.0:
            js *= 1.01
    return js, 0, 0

@njit
def mcmove_core(pos, n, L, rc2, erc, js, ntry, naccept, beta, rng_state):
    if n == 0:
        return ntry, naccept, js
    ntry += 1
    i = lcg_randrange(rng_state, n)
    ox = pos[i, 0]; oy = pos[i, 1]; oz = pos[i, 2]
    eo = particle_energy(pos, n, ox, oy, oz, i, L, rc2, erc)
    nx = ox + (js / L) * (lcg_uniform(rng_state) - 0.5)
    ny = oy + (js / L) * (lcg_uniform(rng_state) - 0.5)
    nz = oz + (js / L) * (lcg_uniform(rng_state) - 0.5)
    en = particle_energy(pos, n, nx, ny, nz, i, L, rc2, erc)
    if lcg_uniform(rng_state) < math.exp(-beta * (en - eo)):
        pos[i, 0] = nx; pos[i, 1] = ny; pos[i, 2] = nz
        naccept += 1
    if ntry % 1000 == 0:
        js, ntry, naccept = adjust_step_core(js, ntry, naccept)
    return ntry, naccept, js

def mcmove(box1, box2, beta, rng_state):
    b = box1 if lcg_bool(rng_state) else box2
    b.ntry, b.naccept, b.js = mcmove_core(
        b.pos, b.n, b.L, b.rc2, b.erc, b.js, b.ntry, b.naccept, beta, rng_state)

# ── Volume move ──

class VolState:
    """vstep: log-volume-ratio move step size, adaptively tuned in mcvol; vtry: volume
    move attempts since the last step-size adjustment; vaccept: volume move
    acceptances since the last step-size adjustment."""
    __slots__ = ("vstep", "vtry", "vaccept")

    def __init__(self, vstep, vtry, vaccept):
        self.vstep, self.vtry, self.vaccept = vstep, vtry, vaccept

@njit
def mcvol_core(pos1, n1, L1, pos2, n2, L2, rc2, erc, Vtot, vstep, vtry, vaccept, beta, rng_state):
    vtry += 1
    v1 = L1 ** 3; v2 = L2 ** 3
    e1o = total_energy_core(pos1, n1, L1, rc2, erc)
    e2o = total_energy_core(pos2, n2, L2, rc2, erc)

    lnv = math.log(v1 / v2) + vstep * (lcg_uniform(rng_state) - 0.5)
    v1n = Vtot * math.exp(lnv) / (1.0 + math.exp(lnv))
    v2n = Vtot - v1n
    L1o, L2o = L1, L2
    L1n = v1n ** (1.0 / 3.0); L2n = v2n ** (1.0 / 3.0)

    e1n = total_energy_core(pos1, n1, L1n, rc2, erc)
    e2n = total_energy_core(pos2, n2, L2n, rc2, erc)
    arg = -beta * ((e1n - e1o) + (e2n - e2o)) + (n1 + 1) * math.log(v1n / v1) + (n2 + 1) * math.log(v2n / v2)

    if lcg_uniform(rng_state) >= math.exp(arg):
        L1n, L2n = L1o, L2o
    else:
        vaccept += 1

    if vtry % 200 == 0:
        ratio = vaccept / vtry
        if ratio < 0.5:
            vstep *= 0.9
        else:
            vstep *= 1.1
        vtry = 0; vaccept = 0

    return L1n, L2n, vstep, vtry, vaccept

def mcvol(e1, e2, Vtot, vs, beta, rng_state):
    e1.L, e2.L, vs.vstep, vs.vtry, vs.vaccept = mcvol_core(
        e1.pos, e1.n, e1.L, e2.pos, e2.n, e2.L, e1.rc2, e1.erc, Vtot,
        vs.vstep, vs.vtry, vs.vaccept, beta, rng_state)

# ── Swap move ──

@njit
def mcswap_core(dst_pos, dst_n, dst_L, src_pos, src_n, src_L, rc2, erc, beta, rng_state):
    v_dst = dst_L ** 3; v_src = src_L ** 3
    tx = lcg_uniform(rng_state) - 0.5
    ty = lcg_uniform(rng_state) - 0.5
    tz = lcg_uniform(rng_state) - 0.5
    e_add = particle_energy(dst_pos, dst_n, tx, ty, tz, -1, dst_L, rc2, erc)
    mu_add = v_dst * math.exp(-beta * e_add) / (dst_n + 1)   # Widom estimator

    if src_n == 0:
        return False, dst_n, src_n, mu_add

    i = lcg_randrange(rng_state, src_n)
    px = src_pos[i, 0]; py = src_pos[i, 1]; pz = src_pos[i, 2]
    e_rm = particle_energy(src_pos, src_n, px, py, pz, i, src_L, rc2, erc)

    arg = -beta * (e_add - e_rm) + math.log(v_dst * src_n / (v_src * (dst_n + 1)))
    if lcg_uniform(rng_state) < math.exp(arg):
        # Remove particle i from src by shifting everything after it down by one
        # (an order-preserving removal, like Julia's deleteat!(src.pos, i)) rather
        # than swapping in the last live particle. A swap-with-last removal would be
        # statistically equivalent (i is drawn uniformly, particles are
        # indistinguishable) but silently permutes which physical particle later
        # RNG-drawn indices land on, which is unnecessary here and only makes the
        # two engines harder to cross-check against each other.
        for m in range(i, src_n - 1):
            src_pos[m, 0] = src_pos[m + 1, 0]
            src_pos[m, 1] = src_pos[m + 1, 1]
            src_pos[m, 2] = src_pos[m + 1, 2]
        dst_pos[dst_n, 0] = tx; dst_pos[dst_n, 1] = ty; dst_pos[dst_n, 2] = tz
        return True, dst_n + 1, src_n - 1, mu_add

    return False, dst_n, src_n, mu_add

def mcswap(e1, e2, mu, beta, rng_state):
    if lcg_bool(rng_state):
        dst, src, dst_i = e1, e2, 0
    else:
        dst, src, dst_i = e2, e1, 1
    accepted, dst.n, src.n, mu_add = mcswap_core(
        dst.pos, dst.n, dst.L, src.pos, src.n, src.L, dst.rc2, dst.erc, beta, rng_state)
    mu[dst_i] += mu_add
    return accepted

# ── Swap-rate adapter ──

class SwapAdapt:
    """n: current swap attempts per adaptation window -- this is what note_swap tunes;
    maxn: upper bound on n; target: target number of accepted swaps per window;
    boundary: relative tolerance around target before n is adjusted; attempts:
    swap attempts so far in the current window; accepted: swap acceptances so far
    in the current window."""
    __slots__ = ("n", "maxn", "target", "boundary", "attempts", "accepted")

    def __init__(self, n, maxn, target, boundary):
        self.n, self.maxn, self.target, self.boundary = n, maxn, target, boundary
        self.attempts, self.accepted = 0, 0

def note_swap(sa, accepted):
    sa.attempts += 1
    if accepted:
        sa.accepted += 1
    if sa.attempts >= max(1, round(sa.n)):
        if sa.accepted > sa.target * (1 + sa.boundary):
            sa.n *= 0.95
        elif sa.accepted < sa.target * (1 - sa.boundary):
            sa.n *= 1.05
        sa.n = min(max(sa.n, 1.0), sa.maxn)
        sa.attempts = 0
        sa.accepted = 0

# ── Output ──
# Per-step trajectory/position dumps aren't used downstream (see Section 1) --
# sample/write_exp below accept out=None/out2=None to skip writing entirely, so
# nothing touches disk (Julia instead routes these to /dev/null; skipping the
# write altogether is the more direct Python equivalent of "discard the output").

def report_stride(rep):
    if rep <= 100:
        return 1
    return 10 ** (int(math.floor(math.log10(rep - 1))) - 1)

def should_report(rep):
    return rep % report_stride(rep) == 0

def sample(out, e1, e2, mu, beta, nsteps, rep):
    if out is None:
        return
    out.write(f"{rep} {e1.rho()} {e2.rho()} {e1.n} {e2.n} {e1.V()} {e2.V()} "
              f"{-math.log(mu[0] / nsteps) / beta} {-math.log(mu[1] / nsteps) / beta} "
              f"{total_energy(e1)} {total_energy(e2)}\n")
    out.flush()

def write_exp(out2, e1, e2, nsteps, do_steps):
    if out2 is None:
        return
    out2.write(f"#L={0.5 * e2.L}; step {nsteps / do_steps}\n")
    for k in range(e2.n):
        x, y, z = wrap(e2.pos[k, 0]), wrap(e2.pos[k, 1]), wrap(e2.pos[k, 2])
        out2.write(f"{x * e2.L} {y * e2.L} {z * e2.L} 0.5 1\n")
    for k in range(e1.n):
        x, y, z = wrap(e1.pos[k, 0]), wrap(e1.pos[k, 1]), wrap(e1.pos[k, 2])
        out2.write(f"{x * e1.L + e2.L + e1.L} {y * e1.L} {z * e1.L} 0.5 2\n")

# ── GEMC run ──

def gemc(e1, e2, Vtot, beta, ntotal, do_steps, nrep, even_out, nequil, outfile, posfile,
         rng_state, *, npart, nvol, nswap0, nswap_max, swap_target, swap_boundary):

    vs = VolState(0.1, 0, 0)
    sa = SwapAdapt(nswap0, nswap_max, swap_target, swap_boundary)
    mu = np.zeros(2)
    nsteps = 0
    s1 = s2 = 0.0
    nacc = 0

    out = None if outfile is None else open(outfile, "w")
    out2 = None if posfile is None else open(posfile, "w")

    while nsteps < do_steps:
        nsteps += 1
        nswap_now = max(1, round(sa.n))
        total = npart + nvol + nswap_now
        R = lcg_randrange(rng_state, total)
        if R < npart:
            mcmove(e1, e2, beta, rng_state)
        elif R < npart + nvol:
            mcvol(e1, e2, Vtot, vs, beta, rng_state)
        else:
            accepted = mcswap(e1, e2, mu, beta, rng_state)
            note_swap(sa, accepted)

        if nsteps % nrep == 0:
            cycle = nsteps // ntotal
            rep = nsteps // nrep
            if even_out or should_report(rep):
                sample(out, e1, e2, mu, beta, nsteps, cycle)
                if cycle > nequil:
                    r1, r2 = sorted((e1.rho(), e2.rho()))
                    s1 += r1; s2 += r2; nacc += 1

    write_exp(out2, e1, e2, nsteps, do_steps)
    if out is not None:
        out.close()
    if out2 is not None:
        out2.close()
    rho1, rho2 = (s1 / nacc, s2 / nacc) if nacc > 0 else (float("nan"), float("nan"))
    return SimpleNamespace(rho1=rho1, rho2=rho2, e1=e1, e2=e2)

# ── Main ──

def main(infile="read.in", outfile="data.out", posfile="p.dat", *,
         npart=3000, nvol=15,
         nswap0=3000.0, nswap_max=10_000.0,
         swap_target=1.0, swap_boundary=0.025):

    # infile holds 9 whitespace-separated values, one per line, in order:
    # T ntotal rho0 v1r rc bs ncycles seed nvol
    with open(infile) as f:
        toks = f.read().split()
    it = iter(toks)
    T       = float(next(it))
    ntotal  = int(next(it))
    rho0    = float(next(it))
    v1r     = float(next(it))
    rc      = float(next(it))
    bs      = int(next(it))
    ncycles = int(next(it))
    seed    = int(next(it))
    nvol    = int(next(it))   # shadows the nvol keyword arg above -- the infile value always wins

    rng_state = np.array([seed], dtype=np.int64)
    N1 = round(ntotal * v1r); N2 = ntotal - N1
    Vtot = ntotal / rho0
    V1 = v1r * Vtot; V2 = Vtot - V1
    L1 = V1 ** (1.0 / 3.0); L2 = V2 ** (1.0 / 3.0)
    n1 = math.ceil(N1 ** (1.0 / 3.0)); n2 = math.ceil(N2 ** (1.0 / 3.0))
    if not (L1 >= n1 and L2 >= n2):
        raise ValueError(f"density too high: L1={L1} n1={n1}, L2={L2} n2={n2}")
    beta = 1.0 / T
    do_steps = ncycles * ntotal
    nequil = ncycles // 2   # average densities over the second half of the run (post-equilibration)
    print("beta=", beta, " v1r=", v1r, " rho0=", rho0)

    e1 = make_box(N1, L1, rc, ntotal)
    e2 = make_box(N2, L2, rc, ntotal)

    # bs < 0: even-interval output every |bs| cycles (ntotal*|bs| steps)
    # bs > 0: log-spaced output, same block size ntotal*bs steps
    even_out = bs < 0
    nrep = ntotal * abs(bs)

    return gemc(e1, e2, Vtot, beta, ntotal, do_steps, nrep, even_out, nequil, outfile, posfile,
                rng_state, npart=npart, nvol=nvol, nswap0=nswap0, nswap_max=nswap_max,
                swap_target=swap_target, swap_boundary=swap_boundary)

## Fitting engine

Each fixed-volume GEMC run at a given initial density only approximates the true
coexistence densities; running the same setup across many initial densities gives a
family of results that can be fit to recover them more precisely. This engine filters
out unreliable points, fits a curve to the dilute branch (the dense branch follows from
mass conservation), and refines the fit by removing outliers. `run_fit(x, y, y2; ...)`
is the entry point, returning the fitted `(x_mean, y_mean, y2_mean)` coexistence densities.

In [ ]:
import re

# Non-uniform-grid gradient, matching numpy.gradient's central-difference scheme --
# the Julia notebook's gradient1d was hand-written specifically to match
# numpy.gradient's algorithm, so here it *is* numpy.gradient.
def gradient1d(y, x):
    return np.gradient(y, x)

def trapz(y, x):
    try:
        return np.trapz(y, x)
    except AttributeError:
        return np.trapezoid(y, x)


def read_initial_params_from_log(log_file):
    """Optional warm start: parse previously fitted parameter sets out of a saved run
    log, for use as _best_opt's initial guesses."""
    params_list = []
    try:
        with open(log_file) as f:
            content = f.read()
        # Match this cell's own "a=..; b=..; c=.." / "d=..; e=..; f=.." print format
        # (see _print_params below).
        abc = re.findall(r"a=([\d.]+);\s*b=([\d.]+);\s*c=([\d.]+)", content)
        def_ = re.findall(r"d=([\d.]+);\s*e=([\d.]+);\s*f=([\d.]+)", content)
        if len(abc) == len(def_):
            for (a, b, c), (d, e, f) in zip(abc, def_):
                params_list.append([float(v) for v in (a, b, c, d, e, f)])
        print(f"Read {len(params_list)} parameter set(s) from {log_file}")
    except Exception as ex:
        print(f"Warning: {ex}")
    return params_list


def print_derivative_table(x, y, y2):
    fwd = np.diff(y) / np.diff(x)
    dy = np.append(fwd, fwd[-1])          # forward diff; last point reuses n-2 value
    dy2 = gradient1d(y2, x)
    d2y2 = gradient1d(dy2, x)
    w = 73
    print("\nDerivative Table [y=forward-diff, y2=gradient]:")
    print("=" * w)
    print(f"{'idx':>3} {'x':>8} {'y':>10} {'y2':>10} {'dy/dx':>12} {'dy2/dx':>12} {'d2y2/dx2':>12}")
    print("-" * w)
    for i in range(len(x)):
        print(f"{i:3d} {x[i]:8.3f} {y[i]:10.6f} {y2[i]:10.6f} "
              f"{dy[i]:12.6f} {dy2[i]:12.6f} {d2y2[i]:12.6f}")
    print("=" * w)
    print("Filter guide: transition = dy/dx sign-change nearest min(y); "
          "lo = first point whose slope no longer exceeds mean+3σ of the negative "
          "slopes before min(y); trim hi from right while d²y2/dx² > 0.")


def filter_near_diagonal(x, y, y2):
    """Drop points where phase separation was incomplete: a much-smaller-than-neighbors
    |y-y2| width means the two boxes hadn't separated from each other when sampled (rho1
    and rho2 came out close instead of distinctly dilute/dense). Runs before filter_data
    so these don't skew the transition/trim logic there."""
    n = len(x)
    keep = np.ones(n, dtype=bool)
    prev = 0   # nearest surviving neighbor before i; walks back past any point already removed
    for i in range(1, n - 1):
        # reject i if its width is less than half the smaller of its two neighbors' widths
        gap = abs(y[i] - y2[i])
        gap_before = abs(y[prev] - y2[prev])
        gap_after = abs(y[i + 1] - y2[i + 1])
        if gap < min(gap_before, gap_after) / 2:
            keep[i] = False
        else:
            prev = i
    removed = n - int(np.sum(keep))
    if removed > 0:
        print(f"Diagonal-proximity filter: removed {removed}/{n} point(s) with |y-y2| "
              f"gap < half its smaller neighbor gap")
    return x[keep], y[keep], y2[keep]   # endpoints 0 and n-1 are always kept; filter_data trims those


def filter_data(x, y, y2):
    """Return (x_filtered, y_filtered, y2_filtered) covering the coexistence region."""
    n = len(x)
    fwd = np.diff(y) / np.diff(x)
    dy = np.append(fwd, fwd[-1])
    d2y2 = gradient1d(gradient1d(y2, x), x)

    # Transition: sign-change in dy nearest to y minimum
    min_idx = int(np.argmin(y))
    trans, best = min_idx, float("inf")
    for i in range(1, n - 1):
        if dy[i - 1] < 0 and dy[i] >= 0:
            d = abs(y[i] - y[min_idx])
            if d < best:
                best, trans = d, i

    # Low start: no negative-slope point -> keep everything. A single negative-slope point
    # -> reject everything before it. Multiple negative-slope points -> take the mean/SD of
    # the negative slopes preceding the y-minimum, then reject from the low end while
    # slope > mean + 3*SD, stopping at the first point that no longer exceeds the threshold.
    neg_idx = [i for i in range(n) if dy[i] < 0]
    lo = 0
    if len(neg_idx) == 1:
        lo = neg_idx[0]
    elif len(neg_idx) > 1:
        preceding = [i for i in neg_idx if i < min_idx]
        if preceding:
            mu = np.mean(dy[preceding])
            sig = np.std(dy[preceding]) if len(preceding) > 1 else abs(mu) * 0.1
            for i in range(n):
                if dy[i] <= mu + 3 * sig:
                    lo = i
                    break

    # High end: trim from right while d^2y2/dx^2 > 0. The very last point can
    # be a false negative (a numerical dip) -- if the point before it is
    # positive, treat the last point as positive too.
    hi = n - 1
    check_start = min(n - 1, trans + 2)
    for i in range(n - 1, check_start - 1, -1):
        positive = d2y2[i] > 0 or (i == n - 1 and n > 1 and d2y2[n - 2] > 0)
        if positive:
            hi = i
        else:
            break

    lo = max(0, min(lo, trans))
    hi = min(n - 1, max(hi, trans))
    print(f"Filtered range: [{lo}, {hi}], keeping {hi - lo + 1}/{n} points")
    return x[lo:hi + 1], y[lo:hi + 1], y2[lo:hi + 1]


def solve_t(x, a, b, c):
    """Solve b*t^2 + (a-x)*t - c = 0 for the positive root."""
    disc = np.maximum((a - x) ** 2 + 4 * b * c, 0.0)
    return (-(a - x) + np.sqrt(disc)) / (2 * b)


# calc_y returns the y branch of the parametric hyperbola x=x(t), y=y(t); callers get the
# y2 branch from mass conservation (rho1 + rho2 = 2*rho0), i.e. y2 = 2x - y.
def calc_y(x, a, b, c, d, e, f):
    t = solve_t(x, a, b, c)
    with np.errstate(divide="ignore", invalid="ignore"):
        val = -d + e * t + f / t
    return np.where(t > 0, val, 100.0)


def goodness_of_fit(y_data, y_fit, n_params=6):
    n = len(y_data)
    ss_res = np.sum((y_data - y_fit) ** 2)
    ss_tot = np.sum((y_data - np.mean(y_data)) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
    adj_r2 = 1.0 - (1.0 - r2) * (n - 1) / max(n - n_params - 1, 1)
    rmse = np.sqrt(ss_res / n)
    return r2, adj_r2, rmse


def print_fit_quality(label, r2, adj_r2, rmse, n, n_params=6):
    print(f"\n{label} (n={n}, params={n_params}):")
    if n <= n_params:
        print("  WARNING: underdetermined (n ≤ n_params) — metrics unreliable")
    print(f"  R²={r2:.6f}  adj.R²={adj_r2:.6f}  RMSE={rmse:.4e}")


def find_outlier(x, y, p):
    """Return index of the point with the largest absolute residual."""
    y_fit = calc_y(x, *p)
    return int(np.argmax(np.abs(y - y_fit)))


def compare_and_report(m1, m2, n1, n2, outlier_resid=None, n_params=6):
    """Print a Round 1 vs Round 2 comparison table; return true if removing the
    outlier is justified."""
    r2_1, ar2_1, rmse_1 = m1
    r2_2, ar2_2, rmse_2 = m2
    d_rmse = (rmse_1 - rmse_2) / rmse_1 * 100

    print(f"\nFit comparison  (n={n1} → n={n2} after removing outlier):")
    print(f"  {'Metric':<12} {'Round 1':>12} {'Round 2':>12} {'Delta':>12}")
    print(f"  {'R²':<12} {r2_1:>12.6f} {r2_2:>12.6f} {r2_2 - r2_1:>+12.6f}")
    print(f"  {'adj.R²':<12} {ar2_1:>12.6f} {ar2_2:>12.6f} {ar2_2 - ar2_1:>+12.6f}")
    print(f"  {'RMSE':<12} {rmse_1:>12.4e} {rmse_2:>12.4e} {d_rmse:>+11.2f}%")

    if n2 <= n_params:
        print(f"  WARNING: Round 2 underdetermined (n={n2} ≤ {n_params}); ΔRMSE≈100% "
              f"expected — coupled t constrains effective DoF.")

    if outlier_resid is not None and rmse_1 > 0:
        nr = abs(outlier_resid) / rmse_1
        frac = outlier_resid ** 2 / (n1 * rmse_1 ** 2)
        print(f"  Outlier |resid|/RMSE={nr:.2f}  error-share={frac * 100:.1f}% "
              f"({frac * n1:.2f}x fair)  [info only]")

    # Adaptive threshold: unlike |resid|/RMSE (which only measures the outlier's own
    # residual), ΔRMSE also catches cases where the outlier was suppressing the whole fit.
    threshold = 200.0 / n2
    justified = d_rmse > threshold
    print(f"  Verdict: {'JUSTIFIED' if justified else 'NOT justified'}  "
          f"(ΔRMSE {d_rmse:+.2f}% > {threshold:.1f}% = 200/n2)")
    return justified


def plot_fit_min(x_raw, y_raw, y2_raw, final_xf, final_yf, final_y2f, final_p, range_info, label):
    """Simple single-panel plot: raw data (cross), accepted-fit data (empty
    circle), the coexistence shaded region, and the x/y/y2 means (solid points)."""
    fig, ax = plt.subplots(figsize=(3.6, 2.4), dpi=150)

    ax.scatter(x_raw, y_raw, marker="x", color="steelblue")
    ax.scatter(x_raw, y2_raw, marker="x", color="darkorange")

    xs_curve = np.linspace(final_xf.min(), final_xf.max(), 500)
    ys_curve = calc_y(xs_curve, *final_p)
    y2s_curve = 2 * xs_curve - ys_curve
    ax.plot(xs_curve, ys_curve, "-", color="steelblue")
    ax.plot(xs_curve, y2s_curve, "-", color="darkorange")

    ax.scatter(final_xf, final_yf, marker="o", facecolors="white", edgecolors="steelblue")
    ax.scatter(final_xf, final_y2f, marker="o", facecolors="white", edgecolors="darkorange")

    if range_info is not None:
        xs, si, ei, xm, ym, y2m, area = range_info
        if si is not None:
            ax.axvspan(xs[si], xs[ei], alpha=0.15, color="green")
            ax.scatter([xm], [ym], marker="o", color="steelblue", edgecolors="black", zorder=5)
            ax.scatter([xm], [y2m], marker="o", color="darkorange", edgecolors="black", zorder=5)

    ax.set_xlabel("ρ'₀")
    ax.set_ylabel("ρ'₁ or ρ'₂")
    ax.set_xlim(0.0, 0.7)
    ax.set_ylim(-0.1, 0.9)
    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.grid(False)
    fig.tight_layout()
    return fig


# ── Private helpers ──────────────────────────────────────────────────────────

def _default_init(xf, yf):
    xr, yr = xf.max() - xf.min(), yf.max() - yf.min()
    return [xf[np.argmin(yf)], xr * 0.5, xr * 0.1, yf.min() * 0.5, yr * 2, yr * 0.2]


def _bounds(xf, yf):
    xr, yr = xf.max() - xf.min(), yf.max() - yf.min()
    lb = [xf.min() * 0.5, 1e-6, 1e-6, 1e-6, 1e-6, 1e-6]
    ub = [xf.max() * 1.5, xr * 2, xr, abs(yf.max()) * 2, yr * 10, yr * 5]
    return lb, ub


def _best_opt(init_list, xf, yf, lb, ub, tol):
    """Try all initial param sets against the bounded fit; return (best_params, best_error)."""
    def model(x, a, b, c, d, e, f):
        return calc_y(x, a, b, c, d, e, f)

    def sse(p):
        return np.sum((yf - calc_y(xf, *p)) ** 2)

    bounds_pairs = list(zip(lb, ub))
    best_p, best_e = None, float("inf")
    print(f"\nTrying {len(init_list)} initial parameter set(s)...")
    for i, init in enumerate(init_list):
        init = np.asarray(init, dtype=float)
        try:
            # Bounded Levenberg-Marquardt (curve_fit) first -- the natural Python
            # analogue of Julia's LsqFit for this bounded curve-fitting problem.
            popt, _ = curve_fit(model, xf, yf, p0=init, bounds=(lb, ub),
                                 xtol=tol, gtol=tol, maxfev=20000)
            err, p = sse(popt), popt
        except Exception:
            try:
                # Falls back to derivative-free bounded Nelder-Mead when the Jacobian is
                # singular (common for rank-deficient rounds, e.g. Round 2 with n <= n_params).
                res = minimize(sse, init, method="Nelder-Mead", bounds=bounds_pairs,
                                options=dict(xatol=tol, fatol=tol, maxiter=5000, maxfev=5000))
                err, p = res.fun, res.x
            except Exception as ex:
                print(f"  Set {i + 1}: failed ({ex})")
                continue
        better = err < best_e
        if better:
            best_e, best_p = err, p
        print(f"  Set {i + 1}: error={err:.6e}{' (best)' if better else ''}")
    if best_p is None:
        raise RuntimeError("All optimization attempts failed")
    return best_p, best_e


def _positive_range(dy, dy2):
    """Index range of the longest segment where both dy>0 and dy2>0."""
    both = (dy > 0) & (dy2 > 0)
    if not np.any(both):
        return None, None
    idx = np.where(both)[0]
    segs = []
    s = idx[0]
    for i in range(1, len(idx)):
        if idx[i] != idx[i - 1] + 1:
            segs.append((s, idx[i - 1]))
            s = idx[i]
    segs.append((s, idx[-1]))
    return max(segs, key=lambda t: t[1] - t[0])


def _print_params(p):
    a, b, c, d, e, f = p
    print(f"a={a:.6f}; b={b:.6f}; c={c:.6f}")
    print(f"d={d:.6f}; e={e:.6f}; f={f:.6f}")


def _print_t_range(xf, p):
    t = solve_t(xf, p[0], p[1], p[2])
    print(f"set tr [{t.min():.3f}:{t.max():.3f}]")


# method is a vestige of the Python prototype's scipy.optimize(method="trust-constr") call;
# _best_opt above is a fixed reimplementation of that approach (bounded curve_fit, falling
# back to Nelder-Mead), so this kwarg is unused -- kept only as a note of where the fit came from.
def run_fit(x_data, y_data, y2_data, label="fit", method="trust-constr", tol=1e-9,
            initial_params_file=None, use_recommendation=False):
    ord_ = np.argsort(x_data)
    x_data, y_data, y2_data = x_data[ord_], y_data[ord_], y2_data[ord_]
    x_raw, y_raw, y2_raw = x_data, y_data, y2_data   # before any filtering, for plotting
    x_data, y_data, y2_data = filter_near_diagonal(x_data, y_data, y2_data)

    print_derivative_table(x_data, y_data, y2_data)
    xf1, yf1, y2f1 = filter_data(x_data, y_data, y2_data)
    lb, ub = _bounds(xf1, yf1)

    init_list = _default_init(xf1, yf1)
    # unused by this notebook's default flow -- fit_temperature() never passes initial_params_file
    if initial_params_file is not None:
        from_log = read_initial_params_from_log(initial_params_file)
        init_list = from_log if from_log else [init_list]
    else:
        init_list = [init_list]

    # ── Round 1 ───────────────────────────────────────────────────────────────
    p1, err1 = _best_opt(init_list, xf1, yf1, lb, ub, tol)
    print(f"\n── Round 1 (error={err1:.6e}) ──")
    _print_params(p1); _print_t_range(xf1, p1)
    yfit1 = calc_y(xf1, *p1)
    m1 = goodness_of_fit(yf1, yfit1)
    print_fit_quality("Round 1 goodness of fit", *m1, len(yf1))

    # ── Outlier detection (largest |residual|) ────────────────────────────────
    widx = find_outlier(xf1, yf1, p1)
    resid1 = yf1 - yfit1
    rmse1 = m1[2]
    print("\nOutlier analysis (largest |residual|):")
    print(f"  {'[i]':<5} {'x':>8} {'y':>12} {'y_fit':>12} {'residual':>12} {'|r|/RMSE':>10}")
    for i, (xi, yi, yfi, ri) in enumerate(zip(xf1, yf1, yfit1, resid1)):
        print(f"  [{i}]  {xi:8.4f} {yi:12.6f} {yfi:12.6f} {ri:12.4e} "
              f"{abs(ri) / rmse1:10.2f}" + (" ←" if i == widx else ""))

    # ── Round 2 ───────────────────────────────────────────────────────────────
    print(f"\nRemoving outlier: index {widx}  x={xf1[widx]:.4f}  y={yf1[widx]:.6f}  "
          f"|resid|={abs(resid1[widx]):.4e}")
    mask = np.ones(len(xf1), dtype=bool); mask[widx] = False
    xf2, yf2, y2f2 = xf1[mask], yf1[mask], y2f1[mask]

    p2, err2 = _best_opt([p1], xf2, yf2, *_bounds(xf2, yf2), tol)
    print(f"\n── Round 2 (error={err2:.6e}) ──")
    _print_params(p2); _print_t_range(xf2, p2)
    yfit2 = calc_y(xf2, *p2)
    m2 = goodness_of_fit(yf2, yfit2)
    print_fit_quality("Round 2 goodness of fit", *m2, len(yf2))

    justified = compare_and_report(m1, m2, len(yf1), len(yf2), outlier_resid=resid1[widx])

    use_r2 = use_recommendation and justified
    if use_r2:
        final_p, final_xf, final_yf, final_y2f = p2, xf2, yf2, y2f2
        print("\n→ Round 2 parameters accepted (use_recommendation=True, justified).")
    else:
        final_p, final_xf, final_yf, final_y2f = p1, xf1, yf1, y2f1
        print("\n→ Round 1 parameters accepted.")
        if justified:
            print("  Note: Round 2 was JUSTIFIED — re-run with use_recommendation=True to apply.")
    _print_params(final_p); _print_t_range(final_xf, final_p)

    # ── Range statistics and area ──────────────────────────────────────────────
    xs = np.linspace(final_xf.min(), final_xf.max(), 500)
    ys = calc_y(xs, *final_p)
    y2s = 2 * xs - ys
    dy = gradient1d(ys, xs)
    dy2 = gradient1d(y2s, xs)
    si, ei = _positive_range(dy, dy2)
    xm = ym = y2m = area = None
    if si is not None:
        xm = np.mean(xs[si:ei + 1])
        ym = calc_y(xm, *final_p); y2m = 2 * xm - ym
        print(f"Range statistics: x_mean={xm:.6f}, y_mean={ym:.6f}, y2_mean={y2m:.6f}")
        diff_ = y2s[si:ei + 1] - ys[si:ei + 1]
        area = trapz(diff_, xs[si:ei + 1])
        print(f"Area between curves: {area:.6f}")
    range_info = (xs, si, ei, xm, ym, y2m, area)

    fig = plot_fit_min(x_raw, y_raw, y2_raw, final_xf, final_yf, final_y2f, final_p, range_info, label)

    return SimpleNamespace(x_mean=xm, y_mean=ym, y2_mean=y2m, x_smooth=xs, y_smooth=ys, y2_smooth=y2s, fig=fig)

## 1. Simulate

In [ ]:
rho0_set = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]   # initial densities to sample
# Trimmed for a reasonable notebook run time: default to a single temperature so a full
# run finishes quickly. Restore the full sweep by uncommenting the line below.
temps     = [0.85]
# temps   = [0.65, 0.75, 0.85, 0.95, 1.05]   # full sweep (5 temps, 45 sims total)
v1r_set = [0.5]   # box-1 volume fraction (of total volume); add more values to sweep it
def vstr(v): return f"{v:.2f}"
def tstr(T): return f"{T:.2f}"

`run_sims` defaults to `true` below, so this notebook runs the fvGEMC simulations (`N=512`,
`10 000` cycles each) over the `v1r_set × rho0_set × temps` set — **9 simulations for the
default single temperature** (T=0.85). They run sequentially, one after another — Python's
GIL means threading wouldn't speed up this CPU-bound loop (a `ProcessPoolExecutor` could give
real parallelism across simulations, but isn't used here for simplicity); the GEMC engine's hot
loop is JIT-compiled with numba so each simulation still finishes in tens of seconds. Each run's
`(rho1, rho2)` pair — the dilute and dense box densities — is kept only in memory, in the `phase`
dict; nothing is written to disk, so re-running this cell always recomputes the simulations from
scratch.

Uncomment the full `temps` list in the cell above to run all 5 temperatures (45 sims total) —
expect roughly 5x longer.

In [ ]:
run_sims = True  # default: run the simulations; change to False to skip them

In [ ]:
def simulate(v1r, rho0, T, ntotal=512, rc=3.0, bs=1, ncycles=10_000, seed=1, nvol=0):
    # ncycles=1e4 for speed; the paper uses 1e5, but 1e4 is likely enough (see paper Fig. 7)
    infile = os.path.join(SCRATCHDIR, "read.in")
    with open(infile, "w") as f:
        f.write(f"{tstr(T)}\n{ntotal}\n{vstr(rho0)}\n{vstr(v1r)}\n{rc}\n{bs}\n{ncycles}\n{seed}\n{nvol}\n")
    return main(infile, None, None)   # per-step trajectory/position dumps aren't used downstream

In [ ]:
phase = {}
log_lines = []
if run_sims:
    todo = [(v1r, rho0, T) for v1r in v1r_set for rho0 in rho0_set for T in temps]
    log_lines.append(f"simulating {len(todo)} (v1r, rho0, T) combo(s) sequentially "
                      f"(Python's GIL means threading wouldn't speed up this CPU-bound loop)...")
    for v1r, rho0, T in todo:
        result = simulate(v1r, rho0, T)
        phase[(v1r, rho0, T)] = result
        log_lines.append(f"  done v1r={vstr(v1r)} rho0={vstr(rho0)} T={tstr(T)}")
    log_lines.append("done.")
else:
    log_lines.append("run_sims is False — phase left empty (nothing to compare/fit against).")
print("\n".join(log_lines))

## 2. Fit

Reshapes the simulation set into one finite-volume family per temperature, then
fits each family to the true coexistence densities.

### Prepare

`v1r_set` and `rho0_set` collapse into one flat family per temperature: every `(v1r, rho0)`
pair contributes an additional `(avg_density, rho1, rho2)` point to that temperature's
fit, the same way varying `rho0` alone did with a single fixed `v1r`.

In [ ]:
def family_curve(T):
    pts = []
    for v1r in v1r_set:
        for rho0 in rho0_set:
            key = (v1r, rho0, T)
            if key not in phase:
                continue
            p = phase[key]
            pts.append(SimpleNamespace(x=(p.rho1 + p.rho2) / 2, y=p.rho1, y2=p.rho2))
    return pts

families = {T: family_curve(T) for T in temps}

### Run the fits

Fits every temperature in `temps` (~1s/temperature), one at a time.

In [ ]:
def fit_temperature(T):
    pts = families[T]
    x  = np.array([p.x for p in pts])
    y  = np.array([p.y for p in pts])
    y2 = np.array([p.y2 for p in pts])
    r = run_fit(x, y, y2, label=f"T{tstr(T)}", use_recommendation=True)
    return SimpleNamespace(T=T, y_mean=r.y_mean, y2_mean=r.y2_mean, x_mean=r.x_mean, fig=r.fig)

In [ ]:
fits = []
for T in temps:
    fits.append(fit_temperature(T))
fits

### Per-temperature fit figures

Each panel: raw finite-volume points (×), the fitted dilute/dense branches, the
accepted-fit points (open circles), and the coexistence point (filled
circle) inside the shaded valid-density interval.

In [ ]:
for f in fits:
    print(f"T = {f.T}")
    display(f.fig)

## 3. Compare

In [ ]:
# From regular GEMC (volume exchange, 1e7 cycles), initial density 0.30.
reference_text = """0.65 0.0025877399877901074 0.8336377060664388
0.75 0.007081976643846621 0.7844437191798171
0.85 0.018394138677198774 0.7376365555365686
0.95 0.03983633225993487 0.676079771288953
1.05 0.07934873075315706 0.6004955675379813
"""
reference = [SimpleNamespace(T=float(f[0]), rho1=float(f[1]), rho2=float(f[2]))
             for f in (line.split() for line in reference_text.strip("\n").split("\n"))]

In [ ]:
fig, ax = plt.subplots(figsize=(3.6, 2.4), dpi=150)

# Reference: dilute + dense branches combined into one series, hollow markers, drawn first.
ref_rho = [g.rho1 for g in reference] + [g.rho2 for g in reference]
ref_T   = [g.T for g in reference] + [g.T for g in reference]
ax.scatter(ref_rho, ref_T, label="GEMC", marker="s", facecolors="white", edgecolors="black")

# fvGEMC+fit: dilute + dense branches combined into one series, solid red circles, drawn on top.
if fits:
    fv_rho = [f.y_mean for f in fits] + [f.y2_mean for f in fits]
    fv_T   = [f.T for f in fits] + [f.T for f in fits]
    ax.scatter(fv_rho, fv_T, label="fvGEMC", marker="o", color="red")

ax.set_xlabel("ρ")
ax.set_ylabel("T")
ax.set_xlim(-0.05, 0.9)
ax.set_ylim(0.6, 1.2)
ax.legend(loc="lower center", frameon=False)
for spine in ax.spines.values():
    spine.set_visible(True)
ax.grid(False)
fig.tight_layout()
fig

In [ ]:
print(f"{'T':<6}{'fvGEMC dilute':<18}{'GEMC dilute':<18}{'fvGEMC dense':<18}{'GEMC dense'}")
for T in temps:
    fT = next((f for f in fits if f.T == T), None)
    gT = next((g for g in reference if g.T == T), None)
    fs = f"{fT.y_mean:.6f}" if fT is not None else "—"
    fd = f"{fT.y2_mean:.6f}" if fT is not None else "—"
    gs = f"{gT.rho1:.6f}" if gT is not None else "—"
    gd = f"{gT.rho2:.6f}" if gT is not None else "—"
    print(f"{tstr(T):<6}{fs:<18}{gs:<18}{fd:<18}{gd}")

The `fvGEMC` columns should closely track their `GEMC` counterparts at each temperature — that agreement demonstrates that the fixed-volume method faithfully reproduces its regular (volume-exchange) counterpart.